# Final Comparison - Autoencoder V1 / V2 vs PatchCore

This notebook compares all anomaly detection methods on the **Capsule** category.

## Goals
- Evaluate AE V1 on Capsule (scores not yet computed)
- Load AE V2 and PatchCore scores from existing JSON files
- Build a unified comparison table
- Visualize ROC curves and AUROC bar chart
- Draw conclusions on reconstruction-based vs feature-based approaches

## 1. Imports

In [ ]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("repo_root:", repo_root)

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, roc_curve
from torch import nn
from torch.utils.data import DataLoader

from mvtec_dataset.mvtec import MvtecAdDataset
from models.autoencoder import AutoEncoder
from models.autoencoder_v2 import AutoEncoderV2
from models.patchcore import (
    MultiScaleFeatureExtractor,
    build_memory_bank,
    random_coreset_sampling,
)
from evaluation.metrics import compute_anomaly_scores, compute_patchcore_scores
from utils.normalization import preprocess_image

print("All imports OK")

## 2. Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 3. Configuration

In [ ]:
CATEGORY = "capsule"
BATCH_SIZE = 16
K_RATIO = 0.01

# Paths
DATA_DIR = Path("../data/mvtec_ad")
METRICS_DIR = Path("../outputs/metrics")
FIGURES_DIR = Path("../outputs/figures")
CHECKPOINT_DIR = Path("../models/checkpoints")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

#  Checkpoint paths
CHECKPOINT_V1 = CHECKPOINT_DIR / "20260426_130047_capsule_autoencoder_v1.pth"
CHECKPOINT_V2 = CHECKPOINT_DIR / "20260426_131357_capsule_autoencoder_v2.pth"
MEMORY_BANK_PATH = Path(
    "../outputs/memory_banks/20260418_134126_capsule_patchcore_multiscale.pt"
)

#  Existing score JSON files
JSON_AE_V2 = METRICS_DIR / "20260418_141735_capsule_autoencoder_v2.json"
JSON_PATCHCORE = METRICS_DIR / "20260418_134126_capsule_patchcore_multiscale.json"

RUN_NAME = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{CATEGORY}_final_comparison"
print("Run:", RUN_NAME)
print("Data:", DATA_DIR)
print("Checkpoint V1:", CHECKPOINT_V1.exists())
print("Checkpoint V2:", CHECKPOINT_V2.exists())
print("Memory bank:", MEMORY_BANK_PATH.exists())

## 4. Data Loading

In [ ]:
test_dataset = MvtecAdDataset(
    root_dir=str(DATA_DIR),
    category=CATEGORY,
    split="test",
    transform=preprocess_image,
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

train_dataset = MvtecAdDataset(
    root_dir=str(DATA_DIR),
    category=CATEGORY,
    split="train",
    transform=preprocess_image,
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_dataset)} images")
print(f"Test : {len(test_dataset)} images")

## 5. Evaluate AE V1

Scores not yet computed - load checkpoint and run evaluation.

In [ ]:
model_v1 = AutoEncoder().to(DEVICE)
model_v1.load_state_dict(torch.load(CHECKPOINT_V1, map_location=DEVICE))
model_v1.eval()
print("AE V1 loaded:", CHECKPOINT_V1.name)

scores_v1: dict[str, float] = {}

for method in ["mean", "max", "topk_mean"]:
    scores, labels = compute_anomaly_scores(
        model=model_v1,
        dataloader=test_loader,
        device=DEVICE,
        method=method,
        k_ratio=K_RATIO,
    )
    auc = roc_auc_score(labels, scores)
    scores_v1[method] = round(float(auc), 4)
    print(f"  AE V1 - {method:<12}: AUROC = {auc:.4f}")

# Save to JSON for future use
json_v1_path = METRICS_DIR / f"{RUN_NAME}_ae_v1.json"
with open(json_v1_path, "w") as f:
    json.dump(
        {
            "category": CATEGORY,
            "model_version": "v1",
            "checkpoint": str(CHECKPOINT_V1),
            "results": scores_v1,
        },
        f,
        indent=4,
    )
print("Saved:", json_v1_path)

## 6. Load AE V2 Scores

In [ ]:
with open(JSON_AE_V2) as f:
    ae_v2_data = json.load(f)

scores_v2 = ae_v2_data["results"]
print("AE V2 scores loaded:")
for method, auc in scores_v2.items():
    print(f"  {method:<12}: AUROC = {auc:.4f}")

## 7. Load PatchCore Scores

In [ ]:
with open(JSON_PATCHCORE) as f:
    pc_data = json.load(f)

scores_pc = pc_data["results"]
print("PatchCore scores loaded:")
for method, auc in scores_pc.items():
    print(f"  {method:<12}: AUROC = {auc:.4f}")

## 8. Comparison Table

In [ ]:
comparison = pd.DataFrame(
    [
        {"method": "AE V1 (reconstruction)", **scores_v1},
        {"method": "AE V2 (reconstruction)", **scores_v2},
        {"method": "PatchCore (feature-based)", **scores_pc},
    ]
).set_index("method")

# Rename columns for display
comparison.columns = ["AUROC mean", "AUROC max", "AUROC topk"]

comparison.style.format("{:.4f}").background_gradient(
    cmap="RdYlGn", vmin=0.5, vmax=1.0
).set_caption(f"Anomaly Detection - Capsule category")

## 9. AUROC Bar Chart

In [ ]:
methods = ["AE V1", "AE V2", "PatchCore"]
mean_scores = [scores_v1["mean"], scores_v2["mean"], scores_pc["mean"]]
max_scores = [scores_v1["max"], scores_v2["max"], scores_pc["max"]]
topk_scores = [scores_v1["topk_mean"], scores_v2["topk_mean"], scores_pc["topk_mean"]]

x = range(len(methods))
width = 0.25
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, ax = plt.subplots(figsize=(9, 5))

for i, (scores, label, color) in enumerate(
    zip(
        [mean_scores, max_scores, topk_scores],
        ["mean", "max", "topk_mean"],
        colors,
    )
):
    offset = (i - 1) * width
    bars = ax.bar(
        [xi + offset for xi in x],
        scores,
        width=width,
        label=label,
        color=color,
        alpha=0.85,
    )
    # Value labels on bars
    for bar, val in zip(bars, scores):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.008,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

ax.axhline(y=0.5, color="red", linestyle="--", linewidth=1, label="Random (0.5)")
ax.set_xticks(list(x))
ax.set_xticklabels(methods, fontsize=11)
ax.set_ylabel("AUROC")
ax.set_ylim(0.0, 1.15)
ax.set_title(f"AUROC Comparison - {CATEGORY}", fontsize=13)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

fig_path = FIGURES_DIR / f"{RUN_NAME}_auroc_comparison.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print("Figure saved:", fig_path)

## 10. ROC Curves

ROC curves require re-computing scores with labels - uses best scoring method (`topk_mean`) for each model.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

#  AE V1
scores, labels = compute_anomaly_scores(
    model=model_v1,
    dataloader=test_loader,
    device=DEVICE,
    method="topk_mean",
    k_ratio=K_RATIO,
)
fpr, tpr, _ = roc_curve(labels, scores)
ax.plot(
    fpr,
    tpr,
    color="#4C72B0",
    linewidth=2,
    label=f'AE V1 (AUC={scores_v1["topk_mean"]:.3f})',
)

#  AE V2
model_v2 = AutoEncoderV2().to(DEVICE)
model_v2.load_state_dict(torch.load(CHECKPOINT_V2, map_location=DEVICE))
model_v2.eval()

scores, labels = compute_anomaly_scores(
    model=model_v2,
    dataloader=test_loader,
    device=DEVICE,
    method="topk_mean",
    k_ratio=K_RATIO,
)
fpr, tpr, _ = roc_curve(labels, scores)
ax.plot(
    fpr,
    tpr,
    color="#DD8452",
    linewidth=2,
    label=f'AE V2 (AUC={scores_v2["topk_mean"]:.3f})',
)

#  PatchCore
extractor = MultiScaleFeatureExtractor().to(DEVICE)
extractor.eval()

saved = torch.load(MEMORY_BANK_PATH, map_location="cpu")
memory_bank = saved["memory_bank"]
memory_bank = random_coreset_sampling(memory_bank, sampling_ratio=0.1)

scores, labels = compute_patchcore_scores(
    feature_extractor=extractor,
    dataloader=test_loader,
    memory_bank=memory_bank,
    device=DEVICE,
    reduction="topk_mean",
    k_ratio=K_RATIO,
)
fpr, tpr, _ = roc_curve(labels, scores)
ax.plot(
    fpr,
    tpr,
    color="#55A868",
    linewidth=2,
    label=f'PatchCore (AUC={scores_pc["topk_mean"]:.3f})',
)

#  Diagonal
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curves - {CATEGORY}")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()

fig_path = FIGURES_DIR / f"{RUN_NAME}_roc_curves.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print("Figure saved:", fig_path)

## 11. Conclusions

### Key takeaway

| Method | Approach | AUROC topk |
|--------|----------|------------|
| AE V1  | Reconstruction | ~0.53 |
| AE V2  | Reconstruction (deeper) | ~0.61 |
| PatchCore | Feature-based (k-NN) | ~0.88 |

### Why reconstruction-based methods struggle

An autoencoder trained only on normal images learns to reconstruct
the general structure of the object. However, a model that reconstructs
**too well** will also reconstruct anomalies - degrading detection.
AE V2 illustrates this: deeper architecture, better reconstruction,
but only marginally better AUROC than V1.

### Why PatchCore performs better

PatchCore does not rely on reconstruction. It builds a memory bank
of normal patch embeddings and detects anomalies by measuring
**local distance** to the nearest normal patch in feature space.
This approach is more sensitive to local defects (cracks, scratches)
and does not suffer from the reconstruction paradox.

### Limits of this comparison

- Evaluated on a single category (Capsule) - results may vary
- AE models trained for 30 epochs only - more epochs may improve slightly
- PatchCore uses random coreset (10%) - greedy coreset would give better results
- Image-level AUROC only - pixel-level AUROC would require ground truth masks